<a href="https://colab.research.google.com/github/FEMYSHIBU/parkinsons-nn-from-scratch/blob/main/parkinsons_nn_from_scratch/notebook/parkinsons_updrs_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print('Dataset loading')

import pandas as pd
import numpy as np

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/telemonitoring/parkinsons_updrs.data"
df = pd.read_csv(url)
print(df.shape)
df.head()

Dataset loading
(5875, 22)


,subject#,age,sex,test_time,motor_UPDRS,total_UPDRS,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,...,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
0,1,72,0,5.6431,28.199,34.398,0.00662,0.000034,0.00401,0.00317,...,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,1,72,0,12.6660,28.447,34.894,0.00300,0.000017,0.00132,0.00150,...,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,1,72,0,19.6810,28.695,35.389,0.00481,0.000025,0.00205,0.00208,...,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,1,72,0,25.6470,28.905,35.810,0.00528,0.000027,0.00191,0.00264,...,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,1,72,0,33.6420,29.187,36.375,0.00335,0.000020,0.00093,0.00130,...,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361


In [ ]:
print("Select features and target")
target_column = 'total_UPDRS'
columns_to_drop = ['sex', 'test_time', 'motor_UPDRS']

X = df.drop(columns=[target_column] + columns_to_drop)
y = df[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature columns:", list(X.columns))
X.describe()

Select features and target
X shape: (5875, 18)
y shape: (5875,)
Feature columns: ['subject#', 'age', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'PPE']


,subject#,age,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,Jitter:DDP,Shimmer,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
count,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000,5875.000000
mean,21.494128,64.804936,0.006154,0.000044,0.002987,0.003277,0.008962,0.034035,0.310960,0.017156,0.020144,0.027481,0.051467,0.032120,21.679495,0.541473,0.653240,0.219589
std,12.372279,8.821524,0.005624,0.000036,0.003124,0.003732,0.009371,0.025835,0.230254,0.013237,0.016664,0.019986,0.039711,0.059692,4.291096,0.100986,0.070902,0.091498
min,1.000000,36.000000,0.000830,0.000002,0.000330,0.000430,0.000980,0.003060,0.026000,0.001610,0.001940,0.002490,0.004840,0.000286,1.659000,0.151020,0.514040,0.021983
25%,10.000000,58.000000,0.003580,0.000022,0.001580,0.001820,0.004730,0.019120,0.175000,0.009280,0.010790,0.015665,0.027830,0.010955,19.406000,0.469785,0.596180,0.156340
50%,22.000000,65.000000,0.004900,0.000035,0.002250,0.002490,0.006750,0.027510,0.253000,0.013700,0.015940,0.022710,0.041110,0.018448,21.920000,0.542250,0.643600,0.205500
75%,33.000000,72.000000,0.006800,0.000053,0.003290,0.003460,0.009870,0.039750,0.365000,0.020575,0.023755,0.032715,0.061735,0.031463,24.444000,0.614045,0.711335,0.264490
max,42.000000,85.000000,0.099990,0.000446,0.057540,0.069560,0.172630,0.268630,2.107000,0.162670,0.167020,0.275460,0.488020,0.748260,37.875000,0.966080,0.865600,0.731730


In [ ]:
print("Group-aware train-test split (keeps each patient entirely in train OR test)")
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(splitter.split(X, y, groups=df['subject#']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Group-aware train-test split (keeps each patient entirely in train OR test)
Train shape: (4623, 18)
Test shape: (1252, 18)


In [ ]:
train_subjects = set(df['subject#'].iloc[train_idx])
test_subjects = set(df['subject#'].iloc[test_idx])
print("Train subjects:", train_subjects)
print("Test subjects:", test_subjects)

Train subjects: {1, 2, 3, 4, 6, 7, 8, 9, 10, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 30, 32, 33, 34, 35, 36, 39, 40, 41, 42}
Test subjects: {5, 37, 38, 11, 12, 26, 28, 29, 31}


In [ ]:
print("Standardize(fit on train only, transform test with same params)")
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



Standardize(fit on train only, transform test with same params)


In [ ]:
print("Type of X_train_scaled:", type(X_train_scaled))
print("Shape of X_train_scaled (rows, columns):", X_train_scaled.shape)
print("Shape of X_test_scaled (rows, columns):", X_test_scaled.shape)
print("Number of dimensions (X_train_scaled):", X_train_scaled.ndim)

Type of X_train_scaled: <class 'numpy.ndarray'>
Shape of X_train_scaled (rows, columns): (4623, 18)
Shape of X_test_scaled (rows, columns): (1252, 18)
Number of dimensions (X_train_scaled): 2


In [ ]:
print("convert targets into numpy arrary and then reshape to column vectors[n,1]")

y_train_array = y_train.values.reshape(-1,1)
y_test_array = y_test.values.reshape(-1,1)

print("Shape of y_train_array:", y_train_array.shape)
print("Shape of y_test_array:", y_test_array.shape)


convert targets into numpy arrary and then reshape to column vectors[n,1]
Shape of y_train_array: (4623, 1)
Shape of y_test_array: (1252, 1)


In [ ]:
# Get the first row
first_row = X_train_scaled[0]

# Count columns from the first row
cols_via_shape = first_row.shape[0]
cols_via_len = len(first_row)

print("First row shape:", first_row.shape)
print("Number of columns (via first row shape):", cols_via_shape)
print("Number of columns (via len of first row):", cols_via_len)

First row shape: (18,)
Number of columns (via first row shape): 18
Number of columns (via len of first row): 18


In [ ]:
print("NeuralNetwork class")

class NeuralNetwork:
  def __init__(self, n_inputs=18, n_hidden=1, n_outputs=1, learning_rate=0.01):

    np.random.seed(0) #setting a seed value for reproducibility

     # Layer 1: input -> hidden (1 neuron)
    self.W1 = np.random.uniform(-1, 1, (n_inputs, n_hidden))
    self.b1 = np.zeros((1, n_hidden)) #np.zeros(shape) creates a NumPy array filled entirely with 0.0 values, in whatever shape you specify.


    # Layer 2: hidden -> output (1 neuron)
    self.W2 = np.random.uniform(-1, 1, (n_hidden, n_outputs))
    self.b2 = np.zeros((1, n_outputs))

    self.learning_rate = learning_rate

  def sigmoid(self, x):
    return 1 / (1 + np.exp(-x))

  def sigmoid_derivative(self, x):
    return x * (1 - x)

  def forward(self, X):
    X = X.astype(float)
    self.Z1 = np.dot(X, self.W1) + self.b1
    self.A1 = self.sigmoid(self.Z1)

    self.Z2 = np.dot(self.A1, self.W2) + self.b2
    self.A2 = self.Z2
    return self.A2

  def backward(self, X, y, output):
    n = X.shape[0]
    output = self.forward(X)

    #Output layer error signal
    error = output - y
    delta2 = error     #sigmoid_derivative(Z2)=1

    #gradients for W2, b2
    dW2 = 1/n * np.dot(self.A1.T, delta2)
    db2 = 1/n * np.sum(delta2, axis=0, keepdims=True)

    #hidden layer error signal
    delta1 = np.dot(delta2, self.W2.T) * self.sigmoid_derivative(self.A1)

    #gradients for W1, b1
    dW1 = 1/n * np.dot(X.T, delta1)
    db1 = 1/n * np.sum(delta1, axis=0, keepdims=True)

    # Update weights (gradient descent)
    self.W2 -= self.learning_rate * dW2
    self.b2 -= self.learning_rate * db2
    self.W1 -= self.learning_rate * dW1
    self.b1 -= self.learning_rate * db1

  def train(self, X, y, training_iterations=15000):
    for iterations in range(15000):
      output = self.forward(X)
      self.backward(X, y, output)

  def compute_loss(self, y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2) #Mean squared error










NeuralNetwork class


In [ ]:
print('Train and evaluate')

NN = NeuralNetwork()
print(NN.W1, NN.b1, NN.W2, NN.b2)
X = X_train_scaled
y = y_train_array
NN.train(X, y, training_iterations=15000)

print("Weights after training:", NN.W1, NN.b1, NN.W2, NN.b2)

y_pred = NN.forward(X_test_scaled)
print(y_pred)


Train and evaluate
[[ 0.09762701]
 [ 0.43037873]
 [ 0.20552675]
 [ 0.08976637]
 [-0.1526904 ]
 [ 0.29178823]
 [-0.12482558]
 [ 0.783546  ]
 [ 0.92732552]
 [-0.23311696]
 [ 0.58345008]
 [ 0.05778984]
 [ 0.13608912]
 [ 0.85119328]
 [-0.85792788]
 [-0.8257414 ]
 [-0.95956321]
 [ 0.66523969]] [[0.]] [[0.5563135]] [[0.]]
Weights after training: [[ 4.39898522]
 [ 5.69337252]
 [-1.55089489]
 [ 0.88351492]
 [-0.22669959]
 [ 1.22352489]
 [-0.18842288]
 [ 0.26947727]
 [-0.73516148]
 [-0.14329047]
 [-0.27244905]
 [ 0.04675222]
 [ 0.22551133]
 [ 0.04017909]
 [-0.42487847]
 [ 0.66112706]
 [ 0.50011754]
 [-0.21356387]] [[-4.68891788]] [[19.15339415]] [[23.75634028]]
[[26.14257905]
 [24.69381687]
 [24.64085508]
 ...
 [41.12440263]
 [41.6452884 ]
 [40.62220015]]


In [ ]:
test_loss = NN.compute_loss(y_test_array, y_pred)
print("Test loss:", test_loss)

RMSE = np.sqrt(test_loss)


Test loss: 154.60048906491852


In [ ]:
baseline_prediction = np.mean(y_train_array)
baseline_mse = np.mean((y_test_array - baseline_prediction) ** 2)
baseline_rmse = np.sqrt(baseline_mse)

print("RMSE:", RMSE)
print("Baseline RMSE:", baseline_rmse)

RMSE: 12.433844500592668
Baseline RMSE: 8.037329032157544


In [ ]:
print(NN.A1.min(), NN.A1.max())

7.960348236472414e-08 0.9997919168405403


In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test_array, y_pred)
print("R^2 score:", r2)

R^2 score: -2.0346185433949877


In [ ]:
print("END")